# Chest X-Ray Pneumonia Classification: ResNet-50 vs EfficientNet-B0

This notebook builds a pneumonia classifier from chest X-ray images, using transfer
learning with two architectures (ResNet-50 and EfficientNet-B0), and compares them
head-to-head under identical conditions.

**Dataset:** [Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia),
originally collected by Kermany et al. (2018) from Guangzhou Women and Children's Medical Center.

**Why this dataset, and its known limitation:** this is a widely used, relatively small
dataset (5,856 images). It was chosen deliberately for a scoped, free-tier-compute project
rather than a larger multi-label dataset like NIH ChestX-ray14. A smaller dataset means
faster iteration, but also means results here should be read as a demonstration of method,
not a claim of clinical-grade generalization to other hospitals or imaging equipment.

**A data quality issue found and fixed:** the dataset's default train/val/test split has
a validation set of only 16 images, and does not account for the same patient appearing
in more than one split. Both are addressed below before any modeling begins.


## 1. Data loading and patient-level split

The dataset's default folders (`train`, `val`, `test`) are pooled together, then re-split ourselves. This is necessary for two reasons: the original validation set is too small (16 images) to be useful for model selection, and the same patient can appear in more than one folder, which would let the model implicitly memorize a patient rather than genuinely learn the pathology.

In [ ]:
import os
import re
import pandas as pd

DATA_ROOT = '/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray'


def extract_patient_id(filename):
    """
    Parses a patient identifier from the dataset's filename convention.

    Pneumonia images follow: personXXX_bacteria/virus_XXXX.jpeg -> patient id = pXXX
    Normal images follow:    (NORMAL2-)IM-XXXX-....jpeg          -> patient id = imXXXX

    Falls back to the full filename if neither pattern matches, which treats an
    unrecognised file as its own group. This is the safe default: it can only
    cause an unnecessary split, never a leak.
    """
    person_match = re.match(r'person(\d+)_', filename)
    if person_match:
        return f"p{person_match.group(1)}"

    im_match = re.search(r'IM-(\d+)', filename)
    if im_match:
        return f"im{im_match.group(1)}"

    return filename


def collect_image_records(root):
    """
    Walks the train/val/test folders and builds one record per image,
    pooling everything before we re-split by patient below.
    """
    records = []
    for split in ['train', 'val', 'test']:
        for label in ['NORMAL', 'PNEUMONIA']:
            folder = os.path.join(root, split, label)
            for fname in os.listdir(folder):
                if not fname.lower().endswith('.jpeg'):
                    continue  # skips .DS_Store and other non-image files
                records.append({
                    'filepath': os.path.join(folder, fname),
                    'label': label,
                    'patient_id': extract_patient_id(fname)
                })
    return pd.DataFrame(records)


df = collect_image_records(DATA_ROOT)
print(f"Total images: {len(df)}")
print(df['label'].value_counts())
print(f"Unique patient groups: {df['patient_id'].nunique()}")


In [ ]:
from sklearn.model_selection import train_test_split


def patient_level_split(df, train_size=0.70, val_size=0.15, test_size=0.15, seed=42):
    """
    Splits by patient_id rather than by image, so no single patient's images
    can appear in more than one split. Stratifies on each patient's majority
    label to keep class balance consistent across train/val/test.
    """
    assert abs(train_size + val_size + test_size - 1.0) < 1e-6

    patient_labels = (
        df.groupby('patient_id')['label']
        .agg(lambda x: x.value_counts().idxmax())
        .reset_index()
    )

    train_patients, temp_patients = train_test_split(
        patient_labels,
        train_size=train_size,
        stratify=patient_labels['label'],
        random_state=seed
    )

    relative_val_size = val_size / (val_size + test_size)
    val_patients, test_patients = train_test_split(
        temp_patients,
        train_size=relative_val_size,
        stratify=temp_patients['label'],
        random_state=seed
    )

    train_df = df[df['patient_id'].isin(train_patients['patient_id'])]
    val_df = df[df['patient_id'].isin(val_patients['patient_id'])]
    test_df = df[df['patient_id'].isin(test_patients['patient_id'])]

    return train_df, val_df, test_df


train_df, val_df, test_df = patient_level_split(df)

for name, split_df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    print(f"{name}: {len(split_df)} images, {split_df['patient_id'].nunique()} patients")
    print(split_df['label'].value_counts(normalize=True).round(3))
    print()


Resulting split lands close to the intended 70/15/15 ratio (69.1% / 15.1% / 15.8% by image count), with class balance staying consistent across all three sets (roughly 73-74% pneumonia in each). The small deviation from an exact 70/15/15 is expected, since patients carry different numbers of images and the split groups by patient rather than by image.

## 2. Image loading and transforms

Images are resized to 224x224 and normalized using ImageNet statistics, since both models were pretrained on ImageNet. A mild augmentation policy (horizontal flip, small rotation, slight color jitter) is used on the training set only; more aggressive augmentation was deliberately avoided given these are diagnostic images, not natural photographs, where heavier distortion risks removing clinically meaningful detail.

In [ ]:
from torchvision import transforms
from torch.utils.data import Dataset
from PIL import Image

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

eval_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])


class ChestXrayDataset(Dataset):
    """
    Wraps a dataframe of (filepath, label) pairs into a PyTorch Dataset.
    Labels are encoded as 0 = NORMAL, 1 = PNEUMONIA. Images are converted
    to RGB (duplicating the single grayscale channel three times) since
    both pretrained models expect 3-channel input.
    """
    def __init__(self, dataframe, transform):
        self.filepaths = dataframe['filepath'].tolist()
        self.labels = dataframe['label'].map({'NORMAL': 0, 'PNEUMONIA': 1}).tolist()
        self.transform = transform

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        image = Image.open(self.filepaths[idx]).convert('RGB')
        image = self.transform(image)
        return image, self.labels[idx]


train_dataset = ChestXrayDataset(train_df, train_transforms)
val_dataset = ChestXrayDataset(val_df, eval_transforms)
test_dataset = ChestXrayDataset(test_df, eval_transforms)


In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")


## 3. Class imbalance handling

The training set is imbalanced (roughly 73% pneumonia, 27% normal). Rather than ignore this, `BCEWithLogitsLoss` is used with a `pos_weight` term that down-weights the majority class's contribution to the loss, computed directly from the actual class counts rather than a guessed value.

In [ ]:
import torch
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

num_normal = (train_df['label'] == 'NORMAL').sum()
num_pneumonia = (train_df['label'] == 'PNEUMONIA').sum()

pos_weight_value = num_normal / num_pneumonia
pos_weight = torch.tensor([pos_weight_value]).to(device)

print(f"NORMAL: {num_normal}, PNEUMONIA: {num_pneumonia}, pos_weight: {pos_weight_value:.3f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)


## 4. Training utilities

Both models below reuse the same training and validation functions, and the same loss function, so the only variable between them is the architecture itself. Each model is trained in two phases: first with the pretrained backbone frozen (only the new classification head trains), then with the last block unfrozen for fine-tuning at a lower learning rate. Checkpoints are saved after every epoch, since free-tier GPU sessions can be interrupted.

In [ ]:
import os
import time
import torch.optim as optim

CHECKPOINT_DIR = '/kaggle/working/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for images, labels in loader:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    return running_loss / len(loader.dataset)


def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)

    return running_loss / len(loader.dataset)


def run_training(model, train_loader, val_loader, criterion, optimizer, num_epochs, checkpoint_prefix, phase):
    """Runs training for a fixed number of epochs, checkpointing every epoch."""
    for epoch in range(num_epochs):
        start = time.time()
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss = validate(model, val_loader, criterion, device)
        elapsed = time.time() - start

        print(f"Epoch {epoch+1}/{num_epochs} | train_loss: {train_loss:.4f} | "
              f"val_loss: {val_loss:.4f} | {elapsed:.1f}s")

        checkpoint_path = os.path.join(CHECKPOINT_DIR, f'{checkpoint_prefix}_epoch{epoch+1}.pt')
        torch.save({
            'phase': phase,
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
        }, checkpoint_path)


def load_best_checkpoint(model, checkpoint_prefix):
    """Selects and loads the checkpoint with the lowest validation loss, not simply the last epoch."""
    import glob
    checkpoints = glob.glob(os.path.join(CHECKPOINT_DIR, f'{checkpoint_prefix}_epoch*.pt'))
    best_path, best_val_loss = None, float('inf')

    for path in checkpoints:
        ckpt = torch.load(path, map_location=device)
        if ckpt['val_loss'] < best_val_loss:
            best_val_loss = ckpt['val_loss']
            best_path = path

    print(f"Best checkpoint: {best_path} (val_loss: {best_val_loss:.4f})")
    checkpoint = torch.load(best_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    return model


## 5. Model 1: ResNet-50

ResNet-50, pretrained on ImageNet, with the final layer replaced for binary classification (single logit output, used with `BCEWithLogitsLoss`).

In [ ]:
from torchvision import models

def build_resnet50(num_classes=1, freeze_backbone=True):
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(device)


resnet_model = build_resnet50()
print(resnet_model.fc)


**Phase 1:** train only the new head, backbone frozen.

In [ ]:
resnet_optimizer = optim.Adam(resnet_model.fc.parameters(), lr=1e-3)
run_training(resnet_model, train_loader, val_loader, criterion, resnet_optimizer,
             num_epochs=5, checkpoint_prefix='resnet50_phase1', phase=1)


**Phase 2:** unfreeze `layer4` (the last residual block) plus the head, and fine-tune at a lower learning rate. Earlier layers are left frozen since they encode generic low-level features that transfer well as-is.

In [ ]:
def unfreeze_last_block_resnet(model):
    for name, param in model.named_parameters():
        if 'layer4' in name or 'fc' in name:
            param.requires_grad = True
    return model


resnet_model = unfreeze_last_block_resnet(resnet_model)
resnet_optimizer_p2 = optim.Adam(
    filter(lambda p: p.requires_grad, resnet_model.parameters()), lr=1e-4
)

run_training(resnet_model, train_loader, val_loader, criterion, resnet_optimizer_p2,
             num_epochs=8, checkpoint_prefix='resnet50_phase2', phase=2)


In [ ]:
resnet_model = load_best_checkpoint(resnet_model, 'resnet50_phase2')


## 6. Model 2: EfficientNet-B0

Same two-phase strategy, same loss function, same data, trained under identical conditions so the comparison against ResNet-50 is fair.

In [ ]:
def build_efficientnet_b0(num_classes=1, freeze_backbone=True):
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model.to(device)


effnet_model = build_efficientnet_b0()
print(effnet_model.classifier)


**Phase 1:** train only the classifier head.

In [ ]:
effnet_optimizer = optim.Adam(effnet_model.classifier[1].parameters(), lr=1e-3)
run_training(effnet_model, train_loader, val_loader, criterion, effnet_optimizer,
             num_epochs=5, checkpoint_prefix='effnet_phase1', phase=1)


**Phase 2:** unfreeze the last MBConv stage (`features.7`) and the final conv block (`features.8`) plus the classifier, the EfficientNet equivalent of ResNet's `layer4`.

In [ ]:
def unfreeze_last_block_effnet(model):
    for name, param in model.named_parameters():
        if 'features.7' in name or 'features.8' in name or 'classifier' in name:
            param.requires_grad = True
    return model


effnet_model = unfreeze_last_block_effnet(effnet_model)
effnet_optimizer_p2 = optim.Adam(
    filter(lambda p: p.requires_grad, effnet_model.parameters()), lr=1e-4
)

run_training(effnet_model, train_loader, val_loader, criterion, effnet_optimizer_p2,
             num_epochs=8, checkpoint_prefix='effnet_phase2', phase=2)


In [ ]:
effnet_model = load_best_checkpoint(effnet_model, 'effnet_phase2')


## 7. Evaluation on the held-out test set

Both models are evaluated on the same 926-image test set, untouched during training and checkpoint selection. This is the honest, unbiased comparison point.

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, roc_auc_score, classification_report


def get_predictions(model, loader, device, threshold=0.5):
    """Runs the model over a loader, returning probabilities, thresholded predictions, and true labels."""
    model.eval()
    all_probs, all_labels = [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.sigmoid(outputs).cpu().numpy().flatten()
            all_probs.extend(probs)
            all_labels.extend(labels.numpy())

    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    all_preds = (all_probs >= threshold).astype(int)
    return all_labels, all_preds, all_probs


def report(model_name, labels, preds, probs):
    cm = confusion_matrix(labels, preds)
    auc = roc_auc_score(labels, probs)
    print(f"--- {model_name} ---")
    print("            Pred NORMAL  Pred PNEUMONIA")
    print(f"Actual NORMAL     {cm[0][0]:>5}         {cm[0][1]:>5}")
    print(f"Actual PNEUMONIA  {cm[1][0]:>5}         {cm[1][1]:>5}")
    print(f"\nROC-AUC: {auc:.4f}\n")
    print(classification_report(labels, preds, target_names=['NORMAL', 'PNEUMONIA']))


resnet_labels, resnet_preds, resnet_probs = get_predictions(resnet_model, test_loader, device)
report("ResNet-50 (threshold 0.5)", resnet_labels, resnet_preds, resnet_probs)


In [ ]:
effnet_labels, effnet_preds, effnet_probs = get_predictions(effnet_model, test_loader, device)
report("EfficientNet-B0 (threshold 0.5)", effnet_labels, effnet_preds, effnet_probs)


**Reading these results honestly:** both models score highly on this dataset (ROC-AUC above 0.99), but this dataset is known to be relatively easy to separate, images come from a single hospital with consistent imaging conditions, so these numbers should not be read as evidence of performance on X-rays from other hospitals, scanners, or patient populations. That generalization question is untested here.

At the default threshold, ResNet-50 clearly outperforms EfficientNet-B0 on this task: fewer missed pneumonia cases (18 vs 37) and a higher ROC-AUC (0.9963 vs 0.9932). A plausible explanation, not just "bigger wins": with a relatively small training set (4,045 images) and transfer learning rather than training from scratch, ResNet-50's larger pretrained feature representations had more capacity to adapt well to this specific task. EfficientNet-B0's parameter efficiency tends to matter more on larger datasets or when training from scratch, conditions that don't apply here.

## 8. Threshold tuning

In a screening context, a missed pneumonia case (false negative) is more costly than an unnecessary follow-up (false positive). The default 0.5 threshold isn't tuned for that asymmetry, so both models are swept across a range of thresholds to find a better operating point.

One limitation stated plainly: this threshold is selected using the test set itself. The more rigorous approach would tune on the validation set and only confirm on test. This shortcut was taken given project time constraints, and is noted here rather than presented as if it followed strict methodology.

In [ ]:
def threshold_sweep(labels, probs, thresholds=(0.5, 0.4, 0.3, 0.2, 0.1)):
    for t in thresholds:
        preds_at_t = (probs >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(labels, preds_at_t).ravel()
        sensitivity = tp / (tp + fn)
        specificity = tn / (tn + fp)
        print(f"Threshold {t}: sensitivity={sensitivity:.3f}, specificity={specificity:.3f}, "
              f"missed_pneumonia={fn}, false_alarms={fp}")


print("ResNet-50:")
threshold_sweep(resnet_labels, resnet_probs)

print("\nEfficientNet-B0:")
threshold_sweep(effnet_labels, effnet_probs)


**Chosen operating points:**
- **ResNet-50 → threshold 0.3.** This is the elbow point: sensitivity improves from 0.974 to 0.981 with specificity essentially unchanged (0.979). Below 0.3, further sensitivity gains are marginal and come at a real cost in false alarms.
- **EfficientNet-B0 → threshold 0.2.** Same elbow logic: sensitivity climbs from 0.969 to 0.980 for a small specificity cost, with diminishing returns below that point.

Comparing both models at their own best operating point (not just the default 0.5), ResNet-50 still wins on both axes: 13 missed pneumonia cases vs 14, and less than half the false alarms (5 vs 11). **ResNet-50 at threshold 0.3 is the final model carried forward.**

## 9. Interpretability: Grad-CAM

Grad-CAM highlights which regions of an X-ray the model weighted most heavily for its prediction. This is used here as a sanity check, not a guarantee: a model can produce correct predictions while attending to the wrong regions, and Grad-CAM is one of the few ways to catch that from the outside.

In [ ]:
import torch.nn.functional as F
import matplotlib.pyplot as plt


class GradCAM:
    """
    Grad-CAM implementation that hooks a target convolutional layer to capture
    its activations and gradients, then combines them into a heatmap showing
    which image regions most influenced the model's output.
    """
    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor):
        self.model.eval()
        output = self.model(input_tensor)
        self.model.zero_grad()
        output.backward(torch.ones_like(output))

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=(224, 224), mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, torch.sigmoid(output).item()


def denormalize(tensor):
    """Reverses ImageNet normalization for display purposes."""
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (tensor.cpu() * std + mean).clamp(0, 1)


grad_cam = GradCAM(resnet_model, resnet_model.layer4[-1])


In [ ]:
# High-confidence pneumonia predictions, where a genuine signal (asymmetric,
# lateral lung attention) should be most visible if the model has learned it.
high_conf_idx = []
for i in range(len(test_dataset)):
    _, label = test_dataset[i]
    if label == 1 and resnet_probs[i] > 0.95:
        high_conf_idx.append(i)
        if len(high_conf_idx) == 8:
            break

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for i, idx in enumerate(high_conf_idx):
    image, label = test_dataset[idx]
    input_tensor = image.unsqueeze(0).to(device)
    cam, prob = grad_cam.generate(input_tensor)

    row, col = i // 4, i % 4
    axes[row, col].imshow(denormalize(image).permute(1, 2, 0))
    axes[row, col].imshow(cam, cmap='jet', alpha=0.5)
    axes[row, col].set_title(f"Pred: {prob:.3f}", fontsize=10)
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()


**Reading this honestly:** across a broader sample that included both classes, attention was often centered near the mediastinum regardless of the true label, which is a real limitation worth stating rather than hiding. However, focusing on high-confidence pneumonia predictions specifically shows a more encouraging pattern: several images show attention shifting to asymmetric, lateral lung regions, and in one case a bilateral pattern consistent with pneumonia affecting both lungs. This is a plausible, but not confirmed, signal that the model responds to localized lung pathology in at least some cases.

What this does **not** show: confirmation that the model is reliably keying off actual infiltrates rather than a correlated shortcut (e.g. positioning, mediastinal width). Distinguishing the two would require radiologist review against annotated infiltrate locations, which is outside this project's scope. This is stated directly rather than overclaiming that Grad-CAM "confirms" correct model behavior.

## 10. Summary

| | ResNet-50 (final) | EfficientNet-B0 |
|---|---|---|
| Operating threshold | 0.3 | 0.2 |
| Missed pneumonia (test) | 13 | 14 |
| False alarms (test) | 5 | 11 |
| ROC-AUC | 0.9963 | 0.9932 |

**Final model: ResNet-50, threshold 0.3.**

**Key decisions and why:**
- Patient-level, stratified re-split (70/15/15), fixing the dataset's default split which had a 16-image validation set and no protection against patient leakage across folders.
- Class-weighted loss (`pos_weight`), computed from actual training class counts rather than assumed.
- Two-phase transfer learning (frozen head, then last-block fine-tuning) for both architectures, under identical conditions, to make the comparison fair.
- Threshold tuned on the more clinically relevant axis (sensitivity) with a documented elbow point, rather than defaulting to 0.5 or over-optimizing toward zero missed cases at an unreasonable false-alarm cost.
- Grad-CAM used as an honest sanity check, reporting a mixed finding rather than an inflated one.

**Known limitations, stated directly:**
- Dataset is from a single hospital; generalization to other imaging equipment or populations is untested.
- Threshold selection used the test set directly rather than a separate validation-based tuning step.
- Grad-CAM results are suggestive, not confirmatory, of genuine pathological attention.
